# 06 - Inference Demo

Demo du doan 1 anh bang checkpoint final.

In [1]:
import json
import sys
from pathlib import Path

import torch
from PIL import Image
from torchvision import transforms

ROOT = Path('..').resolve()
if str(ROOT / 'src') not in sys.path:
    sys.path.append(str(ROOT / 'src'))

from flowers102.models import create_model

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [2]:
with open("/home/nguyenhuynh/Documents/deep-learning/dataset/flower_data/cat_to_name.json", 'r', encoding='utf-8') as f:
    cat_to_name = json.load(f)

num_classes = len(cat_to_name)
model = create_model('efficientnet_b0', num_classes=num_classes, pretrained=False)
ckpt_path = ROOT / 'checkpoints' / 'final_best.pth'
if not ckpt_path.exists():
    ckpt_path = ROOT / 'checkpoints' / 'baseline_best.pth'
print('using checkpoint:', ckpt_path)
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.to(device).eval()

using checkpoint: /home/nguyenhuynh/Documents/deep-learning/quangbinh/cv/checkpoints/baseline_best.pth


EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [4]:
from pathlib import Path
from PIL import Image
import torch
from torchvision import transforms

# 👉 dùng đúng dataset path
data_dir = Path("/home/nguyenhuynh/Documents/deep-learning/dataset/flower_data")

tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

# 👉 chọn 1 class có ảnh (an toàn hơn)
train_dir = data_dir / "train"

# lấy 1 class bất kỳ có dữ liệu
for cls in sorted(train_dir.iterdir()):
    if cls.is_dir() and len(list(cls.glob("*.jpg"))) > 0:
        sample_image = cls
        break

files = sorted(sample_image.glob("*.jpg"))
assert len(files) > 0, f"No images found in {sample_image}"

sample_file = files[0]
print("sample:", sample_file)

# 👉 load ảnh + predict
img = Image.open(sample_file).convert("RGB")
x = tf(img).unsqueeze(0).to(device)

with torch.no_grad():
    probs = torch.softmax(model(x), dim=1)[0]

topk = torch.topk(probs, k=5)

for idx, score in zip(topk.indices.cpu().tolist(), topk.values.cpu().tolist()):
    class_id = str(idx + 1)
    print(class_id, cat_to_name.get(class_id, "unknown"), round(score, 4))

sample: /home/nguyenhuynh/Documents/deep-learning/dataset/flower_data/train/1/image_06734.jpg
1 pink primrose 0.9979
100 blanket flower 0.0009
98 mexican petunia 0.0004
14 spear thistle 0.0004
91 hippeastrum 0.0001


In [ ]:
import requests
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt

url = "https://www.google.com/imgres?q=pink%20primrose&imgurl=https%3A%2F%2Fwww.everwilde.com%2Fmedia%2F%2F0800%2Fresized%2FFOENSPE-A-Showy-Evening-Primrose-Seeds_medium.jpg&imgrefurl=https%3A%2F%2Fwww.everwilde.com%2Fstore%2FOenothera-speciosa-WildFlower-Seed.html%3Fsrsltid%3DAfmBOooKuXdEVljsf4ooyEueIQKaWbAIsoqh_7gRzIbiP7FXdR91Cwtf&docid=10GKC7v83FQ74M&tbnid=RCgZU8XvIgG7lM&vet=12ahUKEwiKgbHmur2TAxWKlK8BHaKDJtsQnPAOegQIFxAB..i&w=566&h=566&hcb=2&ved=2ahUKEwiKgbHmur2TAxWKlK8BHaKDJtsQnPAOegQIFxAB"

response = requests.get(url)
img = Image.open(BytesIO(response.content))

plt.imshow(img)
plt.axis("off")

IndexError: list index out of range